# Inspect Consolidated Runoff Datasets

Load and visualize the two runoff source datasets for January 2000.

Sources (see `catalog/variables.yml` → `runoff`):
- ERA5-Land total runoff (`ro`, m/month)
- GLDAS-2.1 NOAH total runoff (`runoff_total = Qs_acc + Qsb_acc`, kg m-2 = mean of 3-hourly accumulations; multiply by `8 × days_in_month` to get mm/month — see normalized comparison cells below)
- MWBM (Wieczorek et al. 2024, `runoff`, mm/month) — accessed from ScienceBase as a single CF-conformant NetCDF; native mm/month with `cell_methods: time: sum`. No conversion needed before plotting or comparison.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import xarray as xr

DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

TARGET_TIME = "2000-01-15"
TARGET_YEAR = 2000


## Load both datasets

In [ ]:
datasets = {
    "ERA5-Land (ro)": {
        "path": DATASTORE / "era5_land" / "monthly" / f"era5_land_monthly_{TARGET_YEAR}.nc",
        "var": "ro",
        "units": "m/month",
    },
    "GLDAS-2.1 NOAH (runoff_total)": {
        "path": DATASTORE / "gldas_noah_v21_monthly" / "gldas_noah_v21_monthly.nc",
        "var": "runoff_total",
        "units": "kg m-2 (mean of 3-hourly accumulations)",
    },
    "MWBM (ClimGrid, runoff)": {
        "path": DATASTORE / "mwbm_climgrid" / "ClimGrid_WBM.nc",
        "var": "runoff",
        "units": "mm/month",
    },
}

opened = {}
for label, info in datasets.items():
    nc_path = info["path"]
    if not nc_path.exists():
        print(f"SKIP {label}: {nc_path} not found (run fetch first)")
        continue
    ds = xr.open_dataset(nc_path)
    opened[label] = (ds, info)
    print(f"{label}: {list(ds.data_vars)} | time: {ds.time.values[0]} .. {ds.time.values[-1]} | shape: {dict(ds.sizes)}")


## Dataset representations

In [ ]:
for label, (ds, _) in opened.items():
    print(f"{'=' * 60}\n{label}\n{'=' * 60}")
    display(ds)


## Plot January 2000

In [ ]:
n = len(opened)
if n == 0:
    print("No datasets available yet. Run the fetch commands first.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    axes = axes.flatten() if n > 1 else [axes]

    for idx, (label, (ds, info)) in enumerate(opened.items()):
        ax = axes[idx]
        var = info["var"]
        da = ds[var].sel(time=TARGET_TIME, method="nearest")
        actual_time = str(da.time.values)[:10]

        da.plot(ax=ax, cmap="YlGnBu", robust=True)
        ax.set_title(f"{label}\n{actual_time} | {info['units']}", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    for idx in range(len(opened), len(axes)):
        axes[idx].set_visible(False)

    fig.suptitle(f"Runoff — nearest to {TARGET_TIME}", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()


## Normalized comparison (mm/month, ERA5-Land footprint)

Convert both sources to **mm/month** and clip to the **ERA5-Land bounding box** so the panels share a like-for-like geographic and unit basis.

The native units differ:

- ERA5-Land `ro` is monthly accumulated total runoff in **m**. Multiply by 1000 to get mm/month.
- GLDAS-2.1 NOAH `runoff_total` is stored as the **mean of 3-hourly accumulations** in **kg m⁻²**, NOT as a monthly sum. To recover the monthly total in mm, multiply by `8 × days_in_month` (8 three-hour intervals per day). Per NASA GES DISC's GLDAS-2.1 README, this is the documented convention for any variable ending in `_acc`.

ERA5-Land timestamps are end-of-month (e.g. 2000-01-31) and GLDAS timestamps are start-of-month (e.g. 2000-01-01); both refer to the same January accumulation, and `time.sel(method="nearest")` with a mid-month target picks the right month for each.

ERA5-Land is already subset to CONUS plus contributing watersheds, so using its lat/lon range as the clip extent gives the same geographic footprint for both panels and keeps the colour scale dominated by CONUS values rather than tropical maxima from GLDAS's global grid.


In [ ]:
import calendar

import numpy as np
import pandas as pd


def _to_mm_per_month(ds, info, target_time):
    """Convert a source-specific runoff variable to a 2D field in mm/month.

    For GLDAS-2.1 monthly products, the ``_acc`` variables are stored as the
    *mean* of 3-hourly accumulations.  Multiply by ``8 * days_in_month`` to
    recover the monthly total in mm.  ERA5-Land monthly is already a
    monthly sum, so a simple m -> mm conversion is enough.
    """
    var = info["var"]
    units = info["units"]
    da = ds[var].sel(time=target_time, method="nearest")

    if units == "m/month":
        # ERA5-Land ro: m/month -> mm/month
        return da * 1000.0

    if "kg m-2 (mean of 3-hourly accumulations)" in units:
        # GLDAS-NOAH: monthly mean of 3-hourly accumulations.  Convert to monthly
        # total using the picked timestamp's calendar month.
        ts = pd.Timestamp(da.time.values)
        days = calendar.monthrange(ts.year, ts.month)[1]
        return da * 8.0 * days

    if units == "mm/month":
        # MWBM (ClimGrid): native mm/month with cell_methods 'time: sum'
        return da

    raise ValueError(f"Don't know how to normalize units {units!r}")


def _clip_to_bbox(da, lon_range, lat_range):
    """Clip a 2D DataArray to a lon/lat bounding box, handling descending lats."""
    lon_dim = "lon" if "lon" in da.dims else "x"
    lat_dim = "lat" if "lat" in da.dims else "y"
    lat_vals = da[lat_dim].values
    if lat_vals[0] > lat_vals[-1]:  # descending
        lat_slice = slice(lat_range[1], lat_range[0])
    else:
        lat_slice = slice(lat_range[0], lat_range[1])
    return da.sel({lon_dim: slice(*lon_range), lat_dim: lat_slice})


# Reference dataset providing the clip footprint and colour scale
_REF_LABEL = "ERA5-Land (ro)"

if opened:
    # 1) Normalize both sources to mm/month at the target time
    norm = {}
    for label, (ds, info) in opened.items():
        norm[label] = _to_mm_per_month(ds, info, TARGET_TIME)

    # 2) Use ERA5-Land's lat/lon range as the clip footprint
    if _REF_LABEL in norm:
        ref = norm[_REF_LABEL]
        lon_min, lon_max = float(ref.lon.min()), float(ref.lon.max())
        lat_min, lat_max = float(ref.lat.min()), float(ref.lat.max())
        print(f"ERA5-Land footprint: lon [{lon_min:.2f}, {lon_max:.2f}]  lat [{lat_min:.2f}, {lat_max:.2f}]")
        clipped = {label: _clip_to_bbox(da, (lon_min, lon_max), (lat_min, lat_max)) for label, da in norm.items()}
    else:
        print(f"WARN: {_REF_LABEL} not loaded; skipping ERA5-footprint clip")
        clipped = norm

    actual_times = {label: str(da.time.values)[:10] for label, da in clipped.items()}
    for label, da in clipped.items():
        print(f"{label} ({actual_times[label]}): mean = {float(da.mean(skipna=True)):.2f} mm/month (ERA5 footprint)")

    # 3) Shared colour scale derived from ERA5 (land-masked CONUS)
    if _REF_LABEL in clipped:
        ref_vals = clipped[_REF_LABEL].values.ravel()
        ref_vals = ref_vals[~np.isnan(ref_vals)]
        vmin = float(np.percentile(ref_vals, 2))
        vmax = float(np.percentile(ref_vals, 98))
        print(f"Colour scale from {_REF_LABEL}: {vmin:.2f} - {vmax:.2f} mm/month")
    else:
        all_vals = np.concatenate([c.values.ravel() for c in clipped.values()])
        all_vals = all_vals[~np.isnan(all_vals)]
        vmin = float(np.percentile(all_vals, 2))
        vmax = float(np.percentile(all_vals, 98))

    fig, axes = plt.subplots(1, len(clipped), figsize=(16, 6), squeeze=False)
    for idx, (label, da) in enumerate(clipped.items()):
        ax = axes[0, idx]
        da.plot(ax=ax, cmap="YlGnBu", vmin=vmin, vmax=vmax)
        ax.set_title(f"{label}\n{actual_times[label]} | mm/month", fontsize=11)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    fig.suptitle(
        f"Runoff - mm/month, ERA5-Land footprint - {TARGET_TIME}\n"
        f"colour scale from {_REF_LABEL}",
        fontsize=13, y=1.03,
    )
    plt.tight_layout()
    plt.show()
else:
    print("No datasets available yet.")


### Why do ERA5-Land and GLDAS-NOAH runoff differ?

After applying the GLDAS `8 × days_in_month` correction, both sources are in the same ballpark for January 2000 (ERA5 ≈ 10 mm/month, GLDAS ≈ 7 mm/month over the ERA5 footprint), and their spatial patterns are reasonably consistent. They are not identical — both products are *total runoff* from land-surface models (LSMs), but they are built from different schemes, different forcing, and different resolutions.

**Storage convention (the magnitude trap)**

Before the conversion, GLDAS values look ~50× too small. The cause is that GLDAS-2.1 monthly products store `_acc` variables as the **mean of 3-hourly accumulations**, not as monthly sums. The variable carries `cell_methods: time: sum` in the file (because the underlying 3-hourly value IS a 3-hour sum), but the monthly aggregation step takes the average across the month, not the sum. NASA GES DISC's GLDAS-2.1 README documents the `8 × N` rule for any `_acc` variable.

ERA5-Land monthly, by contrast, is built by daily resample → monthly sum (`fetch/era5_land.py:daily_to_monthly`), so each value is already the full month's accumulation in m.

**Algorithm and forcing contrast**

- **ERA5-Land `ro`** is total runoff (`sro + ssro`) from ECMWF's H-TESSEL land surface scheme, driven by the ERA5 reanalysis at native ~9 km resolution (provided here on a 0.1° grid). Snow and soil hydrology are tightly coupled to the ECMWF NWP system.
- **GLDAS-2.1 NOAH `runoff_total`** is `Qs_acc + Qsb_acc` (storm surface runoff + baseflow-groundwater runoff) from NASA's NOAH-2.7 land-surface model, driven by a blended forcing dataset (combining model and observational sources) at 0.25° resolution.

**Why values still diverge after the unit fix**

1. **Different LSM physics.** H-TESSEL and NOAH partition incoming precipitation between infiltration, surface runoff, and sub-surface drainage using different soil-moisture, snow, and frozen-soil parameterisations. Two LSMs forced with the same precipitation will still produce different runoff.

2. **Different forcing precipitation.** ERA5-Land uses ERA5 reanalysis precipitation; GLDAS-2.1 blends NOAA CPC gauge-corrected precipitation with model fields. In winter and over mountains the two precipitation products can disagree by 20-50%, and runoff inherits that disagreement.

3. **Different resolution and topography.** ERA5-Land at ~9 km resolves orographic precipitation and snowmelt patterns that GLDAS at ~25 km cannot. Mountainous regions (Sierra Nevada, Rockies, Cascades, Appalachians) typically show systematically higher ERA5-Land runoff in the wet season.

4. **Timestamp convention.** ERA5-Land timestamps are end-of-month (2000-01-31); GLDAS timestamps are start-of-month (2000-01-01). Both refer to the same January accumulation; `sel(time=..., method="nearest")` with a mid-month target picks the right month for each.

**Calibration target implication**

For NHM, the runoff target uses both products as a *normalised range* so the calibration optimiser is sensitive to relative spatial pattern and seasonality rather than absolute magnitude. The GLDAS-mean convention applies equally across all months and across the time-series, so target normalisation cancels the constant `8 × days_in_month` factor for any month-of-year analysis but NOT across months of different lengths. The aggregation/target builder must therefore apply the conversion (or pull from a properly summed monthly file) before any cross-month comparison.


## Clean up

In [ ]:
for label, (ds, _) in opened.items():
    ds.close()
